# Práctica M26 - Clustering Jerárquico
## Segmentación de Clientes y Recomendación de Productos

En esta práctica se desarrolla un análisis de agrupación utilizando el método de Clustering Jerárquico con el objetivo de segmentar clientes en función de sus patrones de evaluación de productos.

A partir de los segmentos obtenidos, se generan recomendaciones personalizadas para los clientes Salomé, Stephanía y Lydia, basadas en el comportamiento de otros consumidores pertenecientes al mismo clúster.

El análisis incluye:
- Carga y exploración de datos
- Estandarización de variables
- Construcción de dendrograma
- Aplicación del modelo de Clustering Jerárquico
- Interpretación de resultados
- Recomendaciones fundamentadas

### 1. Importación de librerías

En esta sección se importan las librerías necesarias para el análisis.  
Se utilizarán herramientas para manipulación de datos, visualización gráfica y construcción del modelo de Clustering Jerárquico.

In [ ]:
# ==========================================
# Importación de librerías
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
import scipy.cluster.hierarchy as sch

%matplotlib inline

Las librerías cargadas permitirán realizar:

- Manipulación estructurada del dataset.
- Análisis estadístico descriptivo.
- Visualización de patrones de comportamiento.
- Construcción del dendrograma.
- Aplicación del modelo de agrupamiento jerárquico.

## 2. Carga del Dataset

Carga del Dataset y Validación Inicial

Introducción:

En esta sección se carga el archivo Amazon.xlsx en un DataFrame de Pandas con el objetivo de verificar su correcta lectura y validar su estructura dimensional.

In [ ]:
# ==========================================
# Carga del Dataset
# ==========================================

import pandas as pd

df = pd.read_excel("Amazon.xlsx")

print("Dimensiones del dataset:", df.shape)
df.head()

Se importa el archivo Amazon.xlsx en un DataFrame denominado df.
Se imprime la dimensión del conjunto de datos para conocer el número de clientes y productos evaluados, y se visualizan las primeras filas para validar estructura.

##Preparación de la Estructura del Dataset

In [ ]:
# ==========================================
# Preparación de la estructura de datos
# ==========================================

# Renombrar columna de clientes
df = df.rename(columns={"Unnamed: 0": "Cliente"})

# Establecer nombre del cliente como índice
df.set_index("Cliente", inplace=True)

print("Nueva estructura del dataset:")
df.head()

Se renombra la columna que contiene los nombres y se establece como índice para que el modelo trabaje únicamente con variables numéricas.

In [ ]:
# ==========================================
# Análisis estadístico descriptivo
# ==========================================

df.describe()

Se realiza un análisis estadístico descriptivo con el objetivo de comprender la distribución, dispersión y comportamiento general de las variables evaluadas por los clientes.

Este análisis permite identificar diferencias de escala entre variables como "Durabilidad" o "Velocidad Entrega", que presentan valores significativamente mayores en comparación con otras variables como "Precio" o "Valor Educativo".

La presencia de diferentes magnitudes confirma la necesidad de aplicar un proceso de estandarización previo al modelo de Clustering Jerárquico, evitando que variables con mayor rango dominen la formación de los grupos.

In [ ]:
# ==========================================
# Estandarización de variables
# ==========================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(df)

# Convertimos nuevamente a DataFrame para conservar estructura
df_scaled = pd.DataFrame(X_scaled, 
                         columns=df.columns, 
                         index=df.index)

df_scaled.head()

Dado que las variables presentan diferentes escalas y magnitudes, se aplica un proceso de estandarización utilizando StandardScaler.

Este procedimiento transforma cada variable para que tenga media igual a 0 y desviación estándar igual a 1, garantizando que todas las dimensiones contribuyan de manera equilibrada al cálculo de distancias euclidianas utilizadas en el Clustering Jerárquico.

El DataFrame resultante (df_scaled) mantiene la estructura original de clientes y variables, pero ahora en escala normalizada, lo que permite avanzar de forma técnicamente correcta hacia la construcción del dendrograma.

In [ ]:
# ==========================================
# Construcción del Dendrograma
# ==========================================

plt.figure(figsize=(15,7))

dendrogram = sch.dendrogram(
    sch.linkage(df_scaled, method='ward')
)

plt.title("Dendrograma - Clustering Jerárquico")
plt.xlabel("Clientes")
plt.ylabel("Distancia Euclidiana")
plt.xticks([])  # ocultamos nombres por legibilidad

plt.show()

Se construye el dendrograma utilizando el método de Ward, el cual minimiza la varianza intra-cluster en cada etapa de fusión.

El dendrograma permite visualizar la estructura jerárquica de los clientes y analizar los niveles de distancia en los que se forman los grupos.

La interpretación se basa en identificar los mayores saltos verticales en la distancia euclidiana, lo que sugiere un punto óptimo de corte para determinar el número adecuado de clusters.

Este análisis visual será la base para definir el número final de segmentos a utilizar en el modelo.

In [ ]:
# ==========================================
# Aplicación del modelo Agglomerative Clustering
# ==========================================

modelo = AgglomerativeClustering(n_clusters=4, linkage='ward')

clusters = modelo.fit_predict(df_scaled)

# Agregamos los clusters al dataset original
df["Cluster"] = clusters

df.head()

Con base en la interpretación visual del dendrograma, se identificaron cuatro segmentos claramente diferenciados antes del mayor salto en la distancia euclidiana.

Por esta razón, se define el número de clusters en cuatro, permitiendo una segmentación más precisa y coherente con la estructura jerárquica observada.

Cada cliente es asignado a uno de los cuatro grupos, incorporando la variable "Cluster" al DataFrame original para facilitar el análisis comparativo posterior.

In [ ]:
df["Cluster"].value_counts()

Se analiza la distribución de clientes por cluster con el objetivo de evaluar el equilibrio de la segmentación obtenida.

Los resultados muestran cuatro grupos con tamaños diferenciados pero consistentes: el Cluster 2 concentra la mayor cantidad de clientes (36), seguido por el Cluster 1 (24), Cluster 0 (21) y Cluster 3 (19).

La distribución es relativamente balanceada, lo cual indica que la segmentación no generó grupos extremadamente pequeños o atípicos, permitiendo realizar un análisis comparativo sólido entre perfiles de clientes.

In [ ]:
df.groupby("Cluster").mean()

Se calculan los promedios de cada variable por cluster con el fin de identificar patrones de comportamiento distintivos entre los segmentos.

Los resultados permiten observar diferencias claras en variables como "Durabilidad", "Tamano Paquete" y "Velocidad Entrega", lo que evidencia que cada grupo presenta prioridades de evaluación distintas.

Por ejemplo, el Cluster 3 muestra los valores promedio más altos en "Durabilidad" y "Tamano Paquete", sugiriendo un perfil orientado a productos robustos y de mayor dimensión. En contraste, otros clusters presentan mayor valoración en aspectos como "Precio" o "Calidad Producto".

Este análisis constituye la base técnica para formular recomendaciones personalizadas, ya que permite identificar similitudes de comportamiento entre clientes pertenecientes al mismo grupo.

In [ ]:
# ==========================================
# Verificación de nombres de clientes
# ==========================================

df.index.tolist()

Se listan todos los nombres de clientes presentes en el índice del DataFrame con el objetivo de verificar la escritura exacta utilizada en el dataset.

Esta validación es necesaria debido a posibles diferencias en acentuación o variaciones ortográficas entre el enunciado del ejercicio y los nombres reales contenidos en la base de datos.

In [ ]:
# ==========================================
# Identificación de cluster de clientes clave
# ==========================================

df.loc[["Salome", "Stephania", "Lydia"]]

Se identifican los clusters asignados a las clientas Salome, Stephania y Lydia con el objetivo de determinar el segmento al que pertenecen.

Esta clasificación permitirá analizar qué otros clientes comparten patrones de evaluación similares, base fundamental para generar recomendaciones personalizadas sustentadas en comportamiento de grupo.

In [ ]:
# ==========================================
# Clientes similares por cluster
# ==========================================

cluster_salome = df[df["Cluster"] == 1]
cluster_stephania = df[df["Cluster"] == 3]
cluster_lydia = df[df["Cluster"] == 2]

print("Clientes en el mismo cluster que Salome:")
print(cluster_salome.index.tolist())

print("\nClientes en el mismo cluster que Stephania:")
print(cluster_stephania.index.tolist())

print("\nClientes en el mismo cluster que Lydia:")
print(cluster_lydia.index.tolist())

Se identifican los clientes que pertenecen al mismo cluster que Salome, Stephania y Lydia con el propósito de analizar patrones de comportamiento similares dentro de cada segmento.

Al compartir el mismo cluster, estos clientes presentan estructuras de evaluación comparables, lo que permite inferir afinidades en preferencias de producto.

Esta agrupación constituye la base para formular recomendaciones fundamentadas en similitud estadística, alineadas con el enfoque de segmentación conductual del análisis.

### Recomendaciones para Salome

Salome pertenece al Cluster 1, grupo conformado por clientes como Fabian, Frank, Gabriel, Henry, Isabelle, Jacob, Leonard, Matthew, Maya, Sebastian, Susanna y Tamara, entre otros.

Este segmento presenta patrones de evaluación similares, caracterizados por una valoración consistente en variables como Tamano Paquete, Calidad Producto y niveles intermedios de Precio.

Con base en la similitud conductual observada, se recomienda a Salome considerar los productos que han adquirido clientes como Gabriel, Leonard y Sebastian, ya que comparten estructuras de preferencia estadísticamente equivalentes dentro del mismo cluster.

La recomendación se fundamenta en la proximidad jerárquica identificada en el modelo, lo que sugiere alta probabilidad de afinidad en decisiones de compra.

### Recomendaciones para Stephania

Stephania pertenece al Cluster 3, segmento integrado por clientes como Isidore, Joseph, Flavia, Helen, Louise, Markian, Maura, Michael, Monica y Myroslav.

Este grupo se caracteriza por presentar los promedios más elevados en variables como Durabilidad y Tamano Paquete, lo que sugiere un perfil orientado hacia productos robustos, de mayor dimensión y con alta percepción de resistencia.

Se recomienda a Stephania considerar los productos adquiridos por clientes como Michael, Louise y Markian, ya que comparten el mismo patrón estructural de evaluación dentro del cluster.

La recomendación se basa en la homogeneidad estadística del grupo, lo que incrementa la probabilidad de satisfacción ante decisiones de compra similares.

### Recomendaciones para Lydia

Lydia pertenece al Cluster 2, el grupo más numeroso, conformado por clientes como Adam, Anna, Edward, Irene, Ivan, John, Lawrence, Marcel, Martin, Sophia, Stephan y Teresa, entre otros.

Este segmento presenta un perfil diferenciado con énfasis en variables como Durabilidad y Calidad Producto, aunque con niveles moderados en Precio y Velocidad Entrega.

Se recomienda a Lydia considerar los productos que han adquirido clientes como Sophia, Martin y Lawrence, quienes comparten patrones de evaluación altamente similares dentro del mismo cluster.

La proximidad observada en el modelo jerárquico respalda la recomendación, al evidenciar afinidad conductual y preferencias estructurales comparables.

## Conclusión

El análisis de Clustering Jerárquico permitió segmentar a los 100 clientes en cuatro grupos homogéneos con base en sus patrones de evaluación de productos.

La aplicación del método de Ward y la interpretación del dendrograma facilitaron la identificación de una estructura natural de segmentación, evidenciando diferencias claras en prioridades de valoración entre los distintos perfiles.

A partir de esta segmentación, fue posible formular recomendaciones personalizadas para Salome, Stephania y Lydia, fundamentadas en la similitud estadística con otros clientes pertenecientes al mismo cluster.

El enfoque utilizado demuestra cómo la analítica de datos permite transformar patrones históricos de comportamiento en estrategias concretas de recomendación, mejorando la toma de decisiones comerciales mediante segmentación objetiva y cuantitativa.

Este ejercicio evidencia el valor del análisis de agrupamiento como herramienta estratégica para personalización de oferta y optimización de experiencia del cliente.

## Reflexión Final

En esta práctica se aplicó el método de Clustering Jerárquico con el objetivo de segmentar clientes a partir de sus patrones de evaluación de productos. A lo largo del desarrollo, fue posible observar cómo la estandarización de variables resulta fundamental para garantizar que todas las dimensiones contribuyan de manera equilibrada en el cálculo de distancias.

La construcción e interpretación del dendrograma permitió identificar la estructura natural de agrupamiento, definiendo cuatro segmentos claramente diferenciados. Posteriormente, la asignación de clusters facilitó el análisis comparativo entre grupos y la identificación de perfiles de comportamiento específicos.

Uno de los aspectos más relevantes del ejercicio fue comprobar cómo el análisis de datos puede traducirse en decisiones prácticas, como la generación de recomendaciones personalizadas basadas en similitud estadística. Al identificar clientes con patrones de evaluación comparables, se pueden inferir afinidades en preferencias de compra de manera objetiva.

Este ejercicio refuerza la importancia de las técnicas de segmentación dentro del análisis de datos aplicado a contextos comerciales, demostrando que la analítica no solo describe información, sino que también permite generar estrategias accionables fundamentadas en evidencia cuantitativa.

## Recomendación Estratégica Empresarial

A partir de la segmentación obtenida mediante Clustering Jerárquico, se recomienda implementar una estrategia de personalización basada en clusters para optimizar la experiencia del cliente y aumentar la probabilidad de conversión.

Los cuatro segmentos identificados presentan patrones de evaluación diferenciados, lo que permite diseñar estrategias específicas para cada perfil. Por ejemplo, el Cluster 3 muestra una fuerte orientación hacia la Durabilidad y el Tamano Paquete, lo que sugiere sensibilidad hacia productos robustos y de mayor escala. En contraste, otros clusters presentan mayor equilibrio entre Precio, Calidad Producto y Velocidad Entrega.

Desde una perspectiva empresarial, la utilización de esta segmentación permitiría:

- Implementar motores de recomendación basados en similitud de cluster.
- Diseñar campañas de marketing segmentadas por perfil conductual.
- Optimizar inventarios alineando oferta con preferencias dominantes por grupo.
- Mejorar estrategias de cross-selling recomendando productos adquiridos por clientes del mismo segmento.

Este enfoque fortalece la toma de decisiones basada en datos, transformando patrones históricos de evaluación en acciones estratégicas que pueden impactar directamente en la satisfacción del cliente y en el desempeño comercial.

La segmentación conductual obtenida demuestra el potencial de la analítica avanzada como herramienta clave en la optimización de estrategias empresariales centradas en el cliente.

In [ ]:
# ==========================================
# Visualización de clusters en 2D con PCA
# ==========================================

from sklearn.decomposition import PCA

# Reducimos dimensionalidad a 2 componentes
pca = PCA(n_components=2)
X_pca = pca.fit_transform(df_scaled)

# Creamos DataFrame con componentes principales
df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=df.index)
df_pca["Cluster"] = df["Cluster"]

# Gráfica
plt.figure(figsize=(10,7))

for cluster in sorted(df_pca["Cluster"].unique()):
    subset = df_pca[df_pca["Cluster"] == cluster]
    plt.scatter(subset["PC1"], subset["PC2"], label=f"Cluster {cluster}")

plt.title("Visualización de Clusters (PCA 2D)")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.legend()
plt.grid(True)

plt.show()

Se aplica Análisis de Componentes Principales (PCA) con el objetivo de reducir la dimensionalidad del dataset a dos componentes principales, permitiendo visualizar gráficamente la segmentación obtenida.

La proyección en dos dimensiones facilita la interpretación visual de la separación entre clusters, mostrando cómo los grupos identificados mediante Clustering Jerárquico mantienen coherencia estructural incluso tras la reducción dimensional.

La dispersión observada confirma que los cuatro segmentos presentan patrones diferenciados, validando la consistencia del modelo de agrupamiento aplicado.